In [1]:
!pip install -q transformers datasets accelerate scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [2]:
import pandas as pd

train_df = pd.read_csv('/kaggle/input/hatespeechtamil/trainV2.csv')
test_df = pd.read_csv('/kaggle/input/hatespeechtamil/TestV2 - testV2.csv')

label_map = {
    'Non-Abusive': 1,
    'Abusive': 0,
    'abusive': 0
}

train_df['Class'] = train_df['Class'].map(label_map)

print("Label distribution in Training Data:")
print(train_df['Class'].value_counts())

train_df.head()

Label distribution in Training Data:
Class
1    1883
0    1769
Name: count, dtype: int64


,Text,Class
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,1
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,0
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,1
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,0
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,1


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "l3cube-pune/tamil-bert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = Dataset.from_pandas(train_df[['Text', 'Class']])
test_dataset = Dataset.from_pandas(test_df[['Text']])

def tokenize_function(examples):
    return tokenizer(examples["Text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("Class", "labels")
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask"])

print("Tokenization for Tamil BERT complete!")

tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/3652 [00:00<?, ? examples/s]

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

Tokenization for Tamil BERT complete!


In [4]:
import numpy as np

final_dataset = tokenized_train.train_test_split(test_size=0.1, seed=42)

train_ds = final_dataset["train"]
val_ds = final_dataset["test"]

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")

Training samples: 3286
Validation samples: 366


In [5]:
import evaluate
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("L3Cube Model and Metrics ready for training!")

2026-02-09 11:52:28.800816: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770637948.977078      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770637949.029288      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770637949.483838      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770637949.483871      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770637949.483874      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/951M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/tamil-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


L3Cube Model and Metrics ready for training!


In [6]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_tamil_bert",
    learning_rate=2e-5,              
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,
    num_train_epochs=3,               
    weight_decay=0.01,
    eval_strategy="epoch",            
    save_strategy="epoch",
    load_best_model_at_end=True,      
    report_to="none",                 
    fp16=True,                        
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer is ready. Ready for the heavy lifting?")

/tmp/ipykernel_55/2109664941.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer is ready. Ready for the heavy lifting?


In [7]:
# Start the training
trainer.train()

# Save the model locally for inference
trainer.save_model("./l3cube_tamil_bert_best")

print("Training complete! The model has been saved.")

Epoch,Training Loss,Validation Loss,F1
1,No log,0.536662,0.830549
2,No log,0.449642,0.859335
3,0.535600,0.437778,0.839378


Training complete! The model has been saved.


In [8]:
import numpy as np
import pandas as pd

predictions_output = trainer.predict(tokenized_test)

preds = np.argmax(predictions_output.predictions, axis=-1)

output_df = pd.DataFrame({
    'Text': test_df['Text'],
    'Class': preds
})

inverse_map = {1: 'Non-Abusive', 0: 'Abusive'}
output_df['Class'] = output_df['Class'].map(inverse_map)

output_df.to_csv('tamil_bert_predictions.csv', index=False)

print("Success! 'tamil_bert_predictions.csv' is ready for download.")
output_df.head()

Success! 'tamil_bert_predictions.csv' is ready for download.


,Text,Class
0,லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளுகபாட்...,Non-Abusive
1,"இன்னும் கைது பண்ணல... அனைத்து பெற்றோர்களும், க...",Non-Abusive
2,"அப்பா,அம்மா, அந்த இன்டர்வியூ பண்ற வக்கிரம்புடி...",Abusive
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...,Abusive
4,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...,Non-Abusive
